# FEATURE ENGINEERING

In [54]:
import pandas as pd
import numpy as np
import seaborn as sns
import os
import ast

EMB_ACLED = False # Only put True if you have new ACLED data, or if you don't have the embeddings file.

In this file we will do the following:

1. Load the data + set index (sort)
2. Create the target variable
3. Feature engineering (create the features)
4. Visualization

### LOAD THE DATA AND SET INDEX

We are first going to create a final dataset with all the information. We start by loading all the clean datasets, and checking the information in them.

In [55]:
# ACLED
acled = pd.read_csv("../data_clean/acled_clean.csv")
acled = acled.set_index(['iso3', 'month']).sort_index()

# IDMC
idmc = pd.read_csv("../data_clean/idmc_clean.csv")
idmc = idmc.set_index(['iso3', 'month']).sort_index()

# HDX
hdx = pd.read_csv("../data_clean/hdx_clean.csv")
hdx = hdx.set_index(['iso3', 'month']).sort_index()

# ECONAI
econAI = pd.read_csv("../data_clean/econAI_clean.csv")
econAI = econAI.set_index(['iso3', 'month']).sort_index()

# INFORM INDEX
inform = pd.read_csv("../data_clean/inform_clean.csv")
inform = inform.set_index(['iso3', 'month']).sort_index()

In [56]:
datasets = {
    "IDMC": idmc,
    "ACLED": acled,
    "HDX": hdx,
    "EconAI": econAI,
    "INFORM Index": inform
}

def analyze_datasets(datasets_dict):
    analysis = []
    
    for name, df in datasets_dict.items():
       
        temp_df = df.copy()
        temp_df = temp_df.reset_index()
        temp_df['month'] = pd.to_datetime(temp_df['month'])
        
        analysis.append({
            "Dataset": name,
            "Rows": len(temp_df),
            "Countries": temp_df['iso3'].nunique(),
            "Min_Date": temp_df['month'].min().strftime('%Y-%m'),
            "Max_Date": temp_df['month'].max().strftime('%Y-%m'),
            "Avg_Months": round(len(temp_df) / temp_df['iso3'].nunique(), 1)
        })
    
    return pd.DataFrame(analysis)

# Torna a executar-ho amb els teus dataframes
df_info = analyze_datasets(datasets)
print(df_info.to_string(index=False))

     Dataset  Rows  Countries Min_Date Max_Date  Avg_Months
        IDMC  8844         89  2018-01  2026-04        99.4
       ACLED 25102        237  1997-01  2026-03       105.9
         HDX  1209        107  1998-05  2026-03        11.3
      EconAI 35472        182  2010-01  2026-03       194.9
INFORM Index 25212        191  2016-05  2027-04       132.0


In [57]:
def check_temporal_gaps(df):
    df = df.copy().reset_index()
    df['month'] = pd.to_datetime(df['month'])
    df = df.sort_values(['iso3', 'month'])
    
    df['diff'] = df.groupby('iso3')['month'].diff() / pd.Timedelta(days=31)
    
    gaps = df[df['diff'] > 1.1]
    
    if gaps.empty:
        print("No temporal gaps found in the dataset.")
    else:
        print(f"{len(gaps)} temporal gaps found:")
        print(gaps[['iso3', 'month', 'diff']].head(10))
    return gaps

gaps_df = check_temporal_gaps(idmc)
gaps_df = check_temporal_gaps(acled)
gaps_df = check_temporal_gaps(hdx)
gaps_df = check_temporal_gaps(econAI)
gaps_df = check_temporal_gaps(inform)

No temporal gaps found in the dataset.
2486 temporal gaps found:
   iso3      month      diff
2   ABW 2018-09-01  4.935484
3   ABW 2019-02-01  4.935484
5   ABW 2019-07-01  3.935484
9   ABW 2020-03-01  4.903226
10  ABW 2020-05-01  1.967742
11  ABW 2020-07-01  1.967742
13  ABW 2020-10-01  1.967742
15  ABW 2021-02-01  2.967742
17  ABW 2021-09-01  5.935484
18  ABW 2021-11-01  1.967742
965 temporal gaps found:
   iso3      month       diff
2   AFG 2018-02-01  10.870968
3   AFG 2018-05-01   2.870968
5   AFG 2018-09-01   2.967742
6   AFG 2019-03-01   5.838710
7   AFG 2019-09-01   5.935484
8   AFG 2020-04-01   6.870968
9   AFG 2020-08-01   3.935484
11  AFG 2021-04-01   6.838710
14  AFG 2021-10-01   3.935484
16  AFG 2022-06-01   6.838710
No temporal gaps found in the dataset.
No temporal gaps found in the dataset.


Not a surprise that ACLED and HDX Signals have temporal lags, as they only contain information for the date and country where an event or an alarm occur. So it's not a problem to have them.

So, regarning the missing data:

- ACLED: Missing values for an entire country during the period covered by ACLED means that there hasen't been any event in that country, so we can impute them easily. We can also impute gaps, because months with missing values are months without events.
- IDMC: Missing values for an entire country during the period covered by IDMC means that there hasen't been any displacement in that country, so we can impute them easily. We can't impute a gap but yes a whole country.
- HDX Signals: Missing values in the period covered by HDX means that nothing happened there, so can be imputed (even a hole country).
- ECONAI: We can't impute anything.
- GOOGLE TRENDS: We can't impute anything
- INFORM Index: We can't impute anything

Based on that our dataset should contain only the countries contained in the intersection on ECONAI and INFORM index. Regarding the time periods, we need to keep only the periods covered by all the datasets, that is >=2018. Notice that we don't have to restrict the more near months, because even if some of the data sets don't have data of the last month, the target variable is only defined by IDMC data, that is updated daily, so we can have data of the target variable since the present. The thing is that as the target variable is defined looking two months ahead, the last two month will always have Nan in the target variable, and consequently won't be used in the training, so we don't have to restrict the data here (No other dataset have a bigger lag than 2 months).

In [58]:
# 1. Get the strict intersection of countries present in BOTH EconAI and INFORM
econAI_countries = set(econAI.reset_index()['iso3'].unique())
inform_countries = set(inform.index.get_level_values('iso3').unique())

# The target list: countries that MUST be in both datasets
target_countries = econAI_countries.intersection(inform_countries)

print("=" * 70)
print(f"🎯 TARGET COUNTRIES (Intersection of EconAI & INFORM): {len(target_countries)}")
print("=" * 70)

# 2. Check how many countries from each dataset are discarded/lost based on this target
for name, df_actual in datasets.items():
    actual_countries = set(df_actual.reset_index()['iso3'].unique())
    
    # Countries that are in the current dataset but WILL BE LOST 
    # because they are not in our target intersection
    discarded = actual_countries - target_countries
    
    # Countries that this dataset lacks to reach the target (if any)
    missing_from_target = target_countries - actual_countries
    
    print(f"📊 {name}:")
    print(f"   • Total countries in raw file: {len(actual_countries)}")
    print(f"   • Kept for analysis: {len(actual_countries.intersection(target_countries))}")
    
    if len(discarded) > 0:
        print(f"   ❌ Discarded countries (not in intersection): {len(discarded)} {sorted(list(discarded))}")
    else:
        print("   ✅ Perfect! No countries discarded from this dataset.")
        
    if len(missing_from_target) > 0:
        print(f"   ⚠️ Lacks these target countries (will cause NaNs): {sorted(list(missing_from_target))}")
        
    print("-" * 70)

🎯 TARGET COUNTRIES (Intersection of EconAI & INFORM): 176
📊 IDMC:
   • Total countries in raw file: 89
   • Kept for analysis: 86
   ❌ Discarded countries (not in intersection): 3 ['AB9', 'MYT', 'NCL']
   ⚠️ Lacks these target countries (will cause NaNs): ['ALB', 'ARE', 'ARG', 'AUT', 'BEL', 'BGR', 'BHS', 'BLZ', 'BRB', 'BRN', 'BTN', 'BWA', 'CAN', 'CHE', 'CHL', 'CHN', 'CRI', 'CUB', 'CZE', 'DEU', 'DNK', 'DOM', 'DZA', 'ERI', 'ESP', 'EST', 'FIN', 'FJI', 'GAB', 'GEO', 'GNB', 'GNQ', 'GRD', 'GTM', 'GUY', 'HRV', 'HUN', 'IRL', 'ISL', 'JAM', 'JOR', 'JPN', 'KOR', 'KWT', 'LAO', 'LSO', 'LTU', 'LUX', 'LVA', 'MAR', 'MDA', 'MDV', 'MKD', 'MLT', 'MNE', 'MNG', 'MRT', 'MUS', 'MYS', 'NAM', 'NOR', 'NPL', 'NZL', 'OMN', 'PAN', 'POL', 'PRK', 'PRT', 'PRY', 'RWA', 'SAU', 'SEN', 'SGP', 'SRB', 'STP', 'SVK', 'SVN', 'SWE', 'SWZ', 'SYC', 'TKM', 'TLS', 'TON', 'TTO', 'TUN', 'URY', 'UZB', 'VNM', 'VUT', 'WSM']
----------------------------------------------------------------------
📊 ACLED:
   • Total countries in raw file:

In [59]:
print(target_countries)

{'COM', 'GBR', 'PAK', 'BWA', 'BFA', 'STP', 'USA', 'KEN', 'SAU', 'TZA', 'MOZ', 'VEN', 'SLV', 'DOM', 'SSD', 'BEN', 'JAM', 'CUB', 'NOR', 'PRY', 'LSO', 'SVN', 'GNQ', 'MKD', 'JOR', 'BHR', 'RUS', 'TJK', 'ZWE', 'OMN', 'DNK', 'HTI', 'BEL', 'FRA', 'VNM', 'HRV', 'PNG', 'GNB', 'CRI', 'NER', 'IDN', 'CHN', 'BRB', 'POL', 'SVK', 'ETH', 'PRK', 'ARG', 'NPL', 'CHL', 'SWZ', 'THA', 'CMR', 'LVA', 'PHL', 'DJI', 'CAN', 'ALB', 'MDV', 'ITA', 'GEO', 'SLB', 'BRA', 'LTU', 'KHM', 'IRN', 'BGR', 'URY', 'BIH', 'WSM', 'VUT', 'PRT', 'FIN', 'GMB', 'GAB', 'IND', 'CAF', 'BRN', 'CIV', 'LKA', 'TUN', 'UGA', 'CHE', 'ARM', 'UZB', 'TON', 'KOR', 'IRQ', 'GUY', 'MDA', 'BLR', 'ARE', 'BLZ', 'PER', 'MAR', 'ISR', 'BGD', 'NGA', 'MRT', 'SWE', 'AFG', 'MUS', 'CZE', 'MLT', 'IRL', 'JPN', 'ISL', 'AZE', 'SDN', 'DZA', 'HND', 'NAM', 'DEU', 'LUX', 'KWT', 'KAZ', 'ECU', 'LAO', 'LBR', 'GRC', 'COL', 'AUT', 'LBY', 'ERI', 'FJI', 'MMR', 'MLI', 'BTN', 'TCD', 'TUR', 'SRB', 'YEM', 'CYP', 'ESP', 'TTO', 'RWA', 'ZMB', 'NZL', 'ZAF', 'BHS', 'SYR', 'NIC', 'SLE'

Let's do now the final dataset containing the countries in the intersecction of EconAI and INFORM, and the time periods covered by IDMC since 2019.


In [60]:
import pandas as pd

# 1. FUNCTION TO STANDARDIZE INDICES
# Converts any date to the "1st of the month" at 00:00:00 and ensures column names match
def standardize_df(df):
    df = df.reset_index() # Bring the index out to columns
    
    # Ensure the country column is named 'iso3' and is uppercase (in case of hidden spaces)
    if 'iso3' in df.columns:
        df['iso3'] = df['iso3'].astype(str).str.strip().str.upper()
        
    # Force the date to the 1st of the month
    if 'month' in df.columns:
        df['month'] = pd.to_datetime(df['month'])
        df['month'] = df['month'].dt.to_period('M').dt.to_timestamp()
        
    # Recreate the multi-index
    return df.set_index(['iso3', 'month'])

# 2. APPLY THE FUNCTION TO ALL YOUR DATASETS
econAI = standardize_df(econAI)
inform = standardize_df(inform)
acled = standardize_df(acled)
hdx = standardize_df(hdx)
idmc = standardize_df(idmc)

# 3. NOW, CREATE THE SKELETON AND PERFORM THE JOIN
max_month_idmc = idmc.index.get_level_values('month').max()
time_range = pd.date_range(start='2018-01-01', end=max_month_idmc, freq='MS') # 'MS' is Month Start

skeleton_index = pd.MultiIndex.from_product(
    [sorted(list(target_countries)), time_range], 
    names=['iso3', 'month']
)

df_base = pd.DataFrame(index=skeleton_index)

# 4. MERGE EVERYTHING
df = df_base.join(econAI, how="left") \
                  .join(inform, how="left") \
                  .join(acled, how="left") \
                  .join(hdx, how="left") \
                  .join(idmc, how="left") 

# Let's check if it's fixed by looking at the first few rows
print(df.head())

                   risk_3   risk_12  logfat_risk_3  logfat_risk_12  INFORM  \
iso3 month                                                                   
AFG  2018-01-01  0.999608  1.000000       8.331437        9.280007     7.8   
     2018-02-01  0.996514  1.000000       8.295601        9.393767     7.8   
     2018-03-01  0.998218  1.000000       8.339758        9.346226     7.8   
     2018-04-01  0.995082  0.999815       8.383570        9.414723     7.8   
     2018-05-01  0.991237  0.998947       8.342653        9.481498     7.7   

                  VU   CC   HA  fatalities  event_count  ... Battles  \
iso3 month                                               ...           
AFG  2018-01-01  7.1  7.7  8.8      2822.0       1145.0  ...   777.0   
     2018-02-01  7.1  7.7  8.8      1829.0        920.0  ...   555.0   
     2018-03-01  7.1  7.7  8.8      2286.0        976.0  ...   600.0   
     2018-04-01  7.1  7.7  8.8      2794.0       1245.0  ...   825.0   
     2018-05-01  7.1 

In [61]:
df.size, df.shape

(369600, (17600, 21))

In [62]:
print(f"Min month: {df.index.get_level_values('month').min()}")
print(f"Max month: {df.index.get_level_values('month').max()}")
print(f"Number of countries: {df.index.get_level_values('iso3').nunique()}")

Min month: 2018-01-01 00:00:00
Max month: 2026-04-01 00:00:00
Number of countries: 176


In [63]:
nans_por_columna = df.isna().sum()
print(nans_por_columna)

risk_3                          176
risk_12                         176
logfat_risk_3                   176
logfat_risk_12                  176
INFORM                            0
VU                                0
CC                                0
HA                                0
fatalities                     3072
event_count                    3072
notes_acled                    3072
Battles                        3072
Explosions/Remote violence     3072
Protests                       3072
Riots                          3072
Strategic developments         3072
Violence against civilians     3072
hdx_value                     16651
hdx_alert_High concern        16651
hdx_alert_Medium concern      16651
monthly_displacement           9056
dtype: int64


Let's now solve the problem of the missing values:

- Missing values in `notes_acled`, `fatalities`, `event_count`, `Battles`, `Explosions/Remote violence`, `Protests`, `Riots`, `Strategic developments` and `Violence against civilians` correspond to months x countries with no events nor fatalities, so we will put 0 on all of them, and an empty string in notes_acled. 

- Missing values in `monthly_displacement` correspond to countries that hasn't had any displacement in all the covered period, so we are putting a 0.

- Missing values in `hdx_alert_Medium concern` and `hdx_alert_High concern` correspond to the month x country with no hdx alert, so we put 0 in both. Te same for `hdx_value`, so we are putting a 0.

We don't impute the Nan values in risk datasets, because they correspond to the last years and will be dropped before training the model because of the Nans in the target variable.

**NOTE:** Also note that in some datsets we are imputing values of the most recent months with the same assumptions as the gaps in the middle of the data set. This is conceptually incorrect but not in this case, because as we have said non of them has missing data (because is not already updated, so the only case when imputing it it's incorrect) in more than the two last months, and as we have said, these observations will be dropped.

In [64]:
df["notes_acled"] = df["notes_acled"].fillna("")
df["fatalities"] = df["fatalities"].fillna(0)
df["event_count"] = df["event_count"].fillna(0)
df["Battles"] = df["Battles"].fillna(0)
df["Explosions/Remote violence"] = df["Explosions/Remote violence"].fillna(0)
df["Protests"] = df["Protests"].fillna(0)
df["Riots"] = df["Riots"].fillna(0)
df["Strategic developments"] = df["Strategic developments"].fillna(0)
df["Violence against civilians"] = df["Violence against civilians"].fillna(0)
df["hdx_alert_Medium concern"] = df["hdx_alert_Medium concern"].fillna(0)
df["hdx_alert_High concern"] = df["hdx_alert_High concern"].fillna(0)
df["hdx_value"] = df["hdx_value"].fillna(0)
df["monthly_displacement"] = df["monthly_displacement"].fillna(0)

In [65]:
df = df.sort_index()
df.head()

risk_3   risk_12  logfat_risk_3  logfat_risk_12  INFORM  \
iso3 month                                                                   
AFG  2018-01-01  0.999608  1.000000       8.331437        9.280007     7.8   
     2018-02-01  0.996514  1.000000       8.295601        9.393767     7.8   
     2018-03-01  0.998218  1.000000       8.339758        9.346226     7.8   
     2018-04-01  0.995082  0.999815       8.383570        9.414723     7.8   
     2018-05-01  0.991237  0.998947       8.342653        9.481498     7.7   

                  VU   CC   HA  fatalities  event_count  ... Battles  \
iso3 month                                               ...           
AFG  2018-01-01  7.1  7.7  8.8      2822.0       1145.0  ...   777.0   
     2018-02-01  7.1  7.7  8.8      1829.0        920.0  ...   555.0   
     2018-03-01  7.1  7.7  8.8      2286.0        976.0  ...   600.0   
     2018-04-01  7.1  7.7  8.8      2794.0       1245.0  ...   825.0   
     2018-05-01  7.1  7.5  8.7      4264.0       1482.0  ...   975.0   

                 Explosions/Remote violence  Protests  Riots  \
iso3 month                                                     
AFG  2018-01-01                       282.0       9.0    2.0   
     2018-02-01                       299.0      16.0    1.0   
     2018-03-01                       320.0      25.0    0.0   
     2018-04-01                       346.0      29.0    1.0   
     2018-05-01                       444.0      10.0    1.0   

                 Strategic developments  Violence against civilians  \
iso3 month                                                            
AFG  2018-01-01                    37.0                        38.0   
     2018-02-01                    23.0                        26.0   
     2018-03-01                    14.0                        17.0   
     2018-04-01                    22.0                        22.0   
     2018-05-01                    18.0                        34.0   

                 hdx_value  hdx_alert_High concern  hdx_alert_Medium concern  \
iso3 month                                                                     
AFG  2018-01-01        0.0                     0.0                       0.0   
     2018-02-01        1.0                     0.0                       1.0   
     2018-03-01        0.0                     0.0                       0.0   
     2018-04-01        0.0                     0.0                       0.0   
     2018-05-01        2.0                     1.0                       0.0   

                 monthly_displacement  
iso3 month                             
AFG  2018-01-01           7456.583577  
     2018-02-01          13918.956010  
     2018-03-01          15410.272726  
     2018-04-01          17198.881440  
     2018-05-01          39769.719735  

[5 rows x 21 columns]

### CREATE THE TARGET VARIABLE

Our target variable is going to be a variable showing if there's going to be a situation for which the country is elegible for an allocation for the first time, in the next 2 months. To do so, we first need to create the variable allocation-elegible.

We say that a country is elegible to recieve an allocation if the country satisfies the necessary conditions for the CERF to send an allocation: 50,000 new internal displacements over a 3 months period, and there are some displacements at that month.

Our target variable predicts whether in the following two months there's going to be a new situation for whitch the CERF would send allocation. To do so, we first create our target variable as an incidence variable that is 1 if the country is allocation-ellegible at some point in the next 2 months, and then, we put Nan to the target variable if its an alert (so it's == 1) and is not one o two months befor the start of one of these periods. The observations that are Nan are going to be dropped before entering to the model, so that the model will only have information about the months before the crisis, making sure that it focus on learining how variables behave before the crisis, not once that crisis is already happening.

In [66]:
df = df.reset_index()
df = df.sort_values(by=['iso3', 'month'])


df['rolling_3m_displacements'] = (
    df.groupby('iso3')['monthly_displacement']
    .rolling(window=3, min_periods=1)
    .sum()
    .reset_index(level=0, drop=True)
)

df['allocation-eligible'] = (
    (df['rolling_3m_displacements'] >= 50000) & 
    (df['monthly_displacement'] > 0)
).astype(int)

df = df.set_index(['iso3', 'month']).sort_index()

Now we can create our target variable:

In [68]:
# Create the target variable: 1 if conflict in t+1 or t+2, else 0
df["target_2m"] = (
    (df.groupby(level="iso3")["allocation-eligible"].shift(-1) == 1) |
    (df.groupby(level="iso3")["allocation-eligible"].shift(-2) == 1)
).astype(int)

df["start_conflict"] = (
    (df["allocation-eligible"] == 1) & 
    (df.groupby(level="iso3")["allocation-eligible"].shift(1).fillna(0) == 0)
).astype(int)

alert_start = (
    (df.groupby(level="iso3")["start_conflict"].shift(-1) == 1) |
    (df.groupby(level="iso3")["start_conflict"].shift(-2) == 1)
)

df.loc[(df["target_2m"] == 1) & (~alert_start), "target_2m"] = np.nan

print("Total positive targets (conflict in next 2 months):", df["target_2m"].sum())

Total positive targets (conflict in next 2 months): 212.0


In [69]:
df.to_parquet("../data_clean/complete_dataset.parquet")

### FEATURE ENGINEERING

In [16]:
df = pd.read_parquet("../data_clean/complete_dataset.parquet")

In [17]:
df.head()

risk_3   risk_12  logfat_risk_3  logfat_risk_12  INFORM  \
iso3 month                                                                   
AFG  2019-01-01  0.997366  1.000000       8.554694        9.716417     7.7   
     2019-02-01  0.993118  0.997806       8.489241        9.786673     7.7   
     2019-03-01  0.994183  0.997635       8.551225        9.668575     7.7   
     2019-04-01  0.992014  0.997427       8.497727        9.726764     7.7   
     2019-05-01  0.993915  0.998244       8.677351        9.830026     7.8   

                  VU   CC   HA  fatalities  event_count  \
iso3 month                                                
AFG  2019-01-01  7.1  7.5  8.7      3255.0         31.0   
     2019-02-01  7.1  7.5  8.7      2501.0         28.0   
     2019-03-01  7.1  7.5  8.7      3159.0         31.0   
     2019-04-01  7.1  7.5  8.7      2803.0         30.0   
     2019-05-01  7.2  7.5  8.8      3404.0         31.0   

                                                       notes_acled  \
iso3 month                                                           
AFG  2019-01-01  [On 1 January 2019, 10 Taliban militants were ...   
     2019-02-01  [On 01 February 2019, 1 policeman was killed b...   
     2019-03-01  [As reported on March 2, over 24 hours, Afghan...   
     2019-04-01  [Detonation: On 01-April-2019, 1 Taliban milit...   
     2019-05-01  [As reported on 01-May-2019, 19 Taliban milita...   

                  hdx_alert_level     hdx_value  monthly_displacement  \
iso3 month                                                              
AFG  2019-01-01                []      0.000000          10789.172972   
     2019-02-01                []      0.000000           9745.059458   
     2019-03-01  [Medium concern]  35111.111111          66039.172976   
     2019-04-01                []      0.000000          15691.135135   
     2019-05-01                []      0.000000          10789.172972   

                 inform_severity_index  rolling_3m_displacements  \
iso3 month                                                         
AFG  2019-01-01               4.133333              10789.172972   
     2019-02-01               4.400000              20534.232430   
     2019-03-01               3.250000              86573.405407   
     2019-04-01               3.350000              91475.367569   
     2019-05-01               4.500000              92519.481083   

                 allocation-eligible  target_2m  start_conflict  
iso3 month                                                       
AFG  2019-01-01                    0        1.0               0  
     2019-02-01                    0        1.0               0  
     2019-03-01                    1        NaN               1  
     2019-04-01                    1        NaN               0  
     2019-05-01                    1        1.0               0

#### DERIVED HUMANITARIAN IMPACT

In this section we are going to create all the derived variables from teh IDMC and the ACLED datasets, that based on literature and the EDA we have done to these datsets, might be useful for the model to learn the pre-conflic dynamics.

From the IDMC, that only contains the `monthly_displacement` column:

In [ ]:
df['disp_6m_avg'] = df.groupby(level='iso3')['monthly_displacement'].transform(lambda x: x.rolling(6).mean())
df['disp_3m_avg'] = df.groupby(level='iso3')['monthly_displacement'].transform(lambda x: x.rolling(3).mean())
df['monthly_displacement_lag1'] = df.groupby('iso3')['monthly_displacement'].shift(1)
df['monthly_displacement_lag2'] = df.groupby('iso3')['monthly_displacement'].shift(2)


From ACLED, we are going to do two things. First we will create some rolling averages and lags from the `fatalities` column, and then we are going to preprocess the `notes_acled` column so that they can provide some additional information about the events that had happended and whether they can result in internal displacements. 

In [ ]:
df['fat_6m_avg'] = df.groupby(level='iso3')['fatalities'].transform(lambda x: x.rolling(6).mean())
df['fat_3m_avg'] = df.groupby(level='iso3')['fatalities'].transform(lambda x: x.rolling(3).mean())
df['fatalities_lag1'] = df.groupby('iso3')['fatalities'].shift(1)
df['fatalities_lag2'] = df.groupby('iso3')['fatalities'].shift(2)

In [ ]:
df.columns.tolist()

In [ ]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import PCA
from tqdm import tqdm
import os

PATH_CHECKPOINT = "../data_clean/checkpoint_acled_complete.parquet"


# ==========================================
# 1. UNIQUE TEXT PREPARATION
# ==========================================
print("Preparing and exploding the DataFrame...")
df_exploded = df.explode("notes_acled")

# Extract only unique and valid texts to avoid processing duplicates
unique_texts = (
    df_exploded["notes_acled"]
    .dropna()
    .astype(str)
    .str.strip()
)
unique_texts = unique_texts[unique_texts != ""].unique().tolist()


# ==========================================
# 2. HEAVY NLP & FEATURE ENGINEERING (ALL IN ONE)
# ==========================================
if EMB_ACLED:
    print(f"Total unique texts to process: {len(unique_texts)}")
    print("Loading the model and computing raw embeddings...")
    model = SentenceTransformer("all-MiniLM-L6-v2")

    # A. Compute Raw Embeddings (384 dimensions)
    embeddings = model.encode(
        unique_texts,
        batch_size=64,       
        show_progress_bar=True,
        convert_to_numpy=True
    )

    # B. Calculate Target Similarity Scores
    target_phrase = "Civilians fleeing violence, forced displacement, and population seeking refuge"
    target_embedding = model.encode([target_phrase]) 
    print("Calculating semantic similarity per note...")
    similarities = cosine_similarity(embeddings, target_embedding).flatten()

    # C. Build Unique Features Mapping (Saving the raw vector as list)
    df_unique_features = pd.DataFrame({
        "notes_acled": unique_texts,
        "displacement_score": similarities,
        "raw_embedding": list(embeddings)  # 384-dim vector saved natively
    })

    # ==========================================
    # SAVE CHECKPOINT (VECTORS + SIMILARITY)
    # ==========================================
    print("Saving intermediate NLP checkpoint...")
    df_unique_features.to_parquet(PATH_CHECKPOINT, index=False)

else:
    print("Skipping heavy NLP processing, loading checkpoint...")
    if os.path.exists(PATH_CHECKPOINT):
        df_unique_features = pd.read_parquet(PATH_CHECKPOINT)
    else:
        raise FileNotFoundError(f"Checkpoint file not found at {PATH_CHECKPOINT}.")


# ==========================================
# 3. MAP NLP FEATURES BACK TO EXPLODED ROWS
# ==========================================
print("Mapping NLP features back to the exploded rows...")
df_exploded = df_exploded.reset_index()
df_exploded = df_exploded.merge(df_unique_features, on="notes_acled", how="left")


# ==========================================
# 4. FILTER CREATION & BINARY FLAGS
# ==========================================
CRITICAL_THRESHOLD = 0.53
df_exploded["is_displacement_event"] = (df_exploded["displacement_score"] >= CRITICAL_THRESHOLD).astype(int)


# ==========================================
# 5. SMART MONTHLY AGGREGATION (YOUR MEAN + MAX POOL LOGIC)
# ==========================================
print("Grouping and computing monthly textual climate metrics...")

def aggregate_vectors_and_scores(group):
    # Extract valid embeddings for this country-month group
    valid_embs = [e for e in group["raw_embedding"] if e is not None and isinstance(e, np.ndarray)]
    
    # Target similarity aggregations
    disp_max = group["displacement_score"].max()
    disp_mean = group["displacement_score"].mean()
    disp_count = group["is_displacement_event"].sum()
    total_count = group["is_displacement_event"].count()
    
    # If there are no events in this group, return zeros
    if not valid_embs:
        return pd.Series({
            'acled_disp_score_max': 0.0,
            'acled_disp_score_mean': 0.0,
            'acled_disp_events_count': 0,
            'acled_total_events': total_count,
            'mean_vector': [0.0] * 384,
            'max_vector': [0.0] * 384
        })
    
    matrix = np.vstack(valid_embs)
    
    # Your exact logic: Mean Pool and Max Pool (via vector norm)
    mean_pool = np.mean(matrix, axis=0)
    norms = np.linalg.norm(matrix, axis=1)
    max_pool = matrix[np.argmax(norms)]
    
    return pd.Series({
        'acled_disp_score_max': disp_max,
        'acled_disp_score_mean': disp_mean,
        'acled_disp_events_count': disp_count,
        'acled_total_events': total_count,
        'mean_vector': list(mean_pool),
        'max_vector': list(max_pool)
    })

# Apply the heavy aggregation per country and month
df_climate_nlp = (
    df_exploded
    .groupby(["iso3", "month"])
    .apply(aggregate_vectors_and_scores)
    .reset_index()
)
df_climate_nlp['acled_disp_events_ratio'] = df_climate_nlp['acled_disp_events_count'] / df_climate_nlp['acled_total_events']


# ==========================================
# 6. DATE CLEANING & PRE-PREPARATION
# ==========================================
df_climate_nlp['iso3'] = df_climate_nlp['iso3'].astype(str).str.strip().str.upper()
df_climate_nlp['month'] = pd.to_datetime(df_climate_nlp['month'])
df_climate_nlp['month'] = df_climate_nlp['month'].dt.to_period('M').dt.to_timestamp()


# ==========================================
# 7. TRAIN AND APPLY PCA & EXPAND EMBEDDINGS TO COLUMNS
# ==========================================
print("Expanding raw embeddings into 768 individual numeric columns...")
# Convert columns of lists into actual numpy matrices
mean_matrix = np.vstack(df_climate_nlp['mean_vector'].values)
max_matrix = np.vstack(df_climate_nlp['max_vector'].values)

# A. LA MÀGIA: Creem les 384 columnes per al Mean i 384 per al Max
for i in range(384):
    df_climate_nlp[f'emb_acled_mean_{i}'] = mean_matrix[:, i]
    df_climate_nlp[f'emb_acled_max_{i}'] = max_matrix[:, i]

print("Training and applying PCA (5 components for Mean, 5 components for Max)...")
# PCA for Mean Pool
pca_mean = PCA(n_components=5, random_state=42)
res_mean = pca_mean.fit_transform(mean_matrix)

# PCA for Max Pool
pca_max = PCA(n_components=5, random_state=42)
res_max = pca_max.fit_transform(max_matrix)

# Add the PCA features directly to the aggregated dataset
for i in range(5):
    df_climate_nlp[f'pca_acled_mean_{i}'] = res_mean[:, i]
    df_climate_nlp[f'pca_acled_max_{i}'] = res_max[:, i]

# Build the 768 concatenated vector list (just in case)
df_climate_nlp['notes_vector_acled'] = df_climate_nlp.apply(
    lambda r: r['mean_vector'] + r['max_vector'], axis=1
)

# Clean up temporary vector columns to save space, keeping indices ready
df_climate_nlp = df_climate_nlp.drop(columns=['mean_vector', 'max_vector'])
df_climate_nlp = df_climate_nlp.set_index(['iso3', 'month']).sort_index()

# Save the features block to Parquet (ara ocuparà una mica més per tenir tantes columnes)
df_climate_nlp.to_parquet("../data_clean/clima_acled.parquet")


# ==========================================
# 8. THE FINAL JOIN WITH YOUR BASIC DATASET
# ==========================================
print("Merging everything into your basic dataset...")
df_features = pd.read_parquet("../data_clean/clima_acled.parquet")

# df contains your basic data. df_features contains Similitud + PCA + 768 Embeddings
df_final = df.join(df_features)
df_final = df_final.sort_index(level=['iso3', 'month'])

# DINÀMIC: Busquem totes les columnes numèriques que acabem de crear per omplir els mesos buits amb 0
numeric_cols_to_fill = [col for col in df_features.columns if col != 'notes_vector_acled']

print(f"Filling missing values with zeros for {len(numeric_cols_to_fill)} columns...")
df_final[numeric_cols_to_fill] = df_final[numeric_cols_to_fill].fillna(0)

# Fill missing raw vectors with a 768-dimensional zero list (for the backup column)
zero_vector_768 = [0.0] * 768
df_final['notes_vector_acled'] = df_final['notes_vector_acled'].apply(
    lambda x: x if isinstance(x, list) else zero_vector_768
)



In [ ]:
df_final.columns.tolist()

#### RISK ALARMS

In this section we are going to create the derived features from EconAI and the HDX Signals. From EconAI we have `risk_3`, `risk_12`, `logfat_risk_3` and `logfat_risk_12`. From HDX Signals we have `hdx_alert_level` and `hdx_value`. 

In [ ]:
features_risk_alerts = [ 'hdx_value', 'hdx_med_high_count', 
                        'hdx_3m_sum', 'hdx_alert_max', 
                        'hdx_alert_sum', 'hdx_alert_mean', 'hdx_medium_count', 'hdx_high_count']

Regardinc EconAI:

In [ ]:
df_final['risk3_6m_avg'] = df_final.groupby(level='iso3')['risk_3'].transform(lambda x: x.rolling(6).mean())
df_final['risk3_3m_avg'] = df_final.groupby(level='iso3')['risk_3'].transform(lambda x: x.rolling(3).mean())
df_final['risk3_lag1'] = df_final.groupby('iso3')['risk_3'].shift(1)
df_final['risk3_lag2'] = df_final.groupby('iso3')['risk_3'].shift(2)

df_final['risk12_6m_avg'] = df_final.groupby(level='iso3')['risk_12'].transform(lambda x: x.rolling(6).mean())
df_final['risk12_3m_avg'] = df_final.groupby(level='iso3')['risk_12'].transform(lambda x: x.rolling(3).mean())
df_final['risk12_lag1'] = df_final.groupby('iso3')['risk_12'].shift(1)
df_final['risk12_lag2'] = df_final.groupby('iso3')['risk_12'].shift(2)

df_final['logfat_risk3_6m_avg'] = df_final.groupby(level='iso3')['logfat_risk_3'].transform(lambda x: x.rolling(6).mean())
df_final['logfat_risk3_3m_avg'] = df_final.groupby(level='iso3')['logfat_risk_3'].transform(lambda x: x.rolling(3).mean())
df_final['logfat_risk3_lag1'] = df_final.groupby('iso3')['logfat_risk_3'].shift(1)
df_final['logfat_risk3_lag2'] = df_final.groupby('iso3')['logfat_risk_3'].shift(2)

df_final['logfat_risk12_6m_avg'] = df_final.groupby(level='iso3')['logfat_risk_12'].transform(lambda x: x.rolling(6).mean())
df_final['logfat_risk12_3m_avg'] = df_final.groupby(level='iso3')['logfat_risk_12'].transform(lambda x: x.rolling(3).mean())
df_final['logfat_risk12_lag1'] = df_final.groupby('iso3')['logfat_risk_12'].shift(1)
df_final['logfat_risk12_lag2'] = df_final.groupby('iso3')['logfat_risk_12'].shift(2)

df_final['risk_gt_06'] = df_final['risk_3'] > 0.6

df_final['logfat_risk_3_gt_5'] = df_final['logfat_risk_3'] > 5 # ESTO JUSTIFICAR CON EDA!!!!! O CAMBIAR EL VALOR A MAYOR O MENOR, NO SE QUE VALORES HAY!


And then, from the HDX Signals:


In [ ]:
# ORDINAL ENCODING for HDX Signals because they have a clear severity order (None < Medium concern < High concern)
hdx_mapping = {
    'None': 0,
    'Medium concern': 1,
    'High concern': 2
}

def get_max_hdx(val_list):
    scores = [hdx_mapping.get(str(x), 0) for x in val_list]
    return max(scores) if scores else 0

def get_sum_hdx(val_list):
    scores = [hdx_mapping.get(str(x), 0) for x in val_list]
    return sum(scores) if scores else 0

def get_mean_hdx(val_list):
    scores = [hdx_mapping.get(str(x), 0) for x in val_list]
    return np.mean(scores) if scores else 0.0

def get_medium_count_hdx(val_list):
    scores = [hdx_mapping.get(str(x), 0) for x in val_list]
    return scores.count(1)

def get_high_count_hdx(val_list):
    scores = [hdx_mapping.get(str(x), 0) for x in val_list]
    return scores.count(2)

def count_hdx_med_high(x):
    if isinstance(x, list):
        return sum(1 for item in x if item in ['Medium concern', 'High concern'])
    elif isinstance(x, str): # In case pandas read the list as a string
        return 1 if 'Medium concern' in x or 'High concern' in x else 0
    return 0

# We will create three different features: max, sum, and mean of the HDX alert levels for each month

df_final['hdx_alert_max'] = df_final['hdx_alert_level'].apply(get_max_hdx)
df_final['hdx_alert_sum'] = df_final['hdx_alert_level'].apply(get_sum_hdx)
df_final['hdx_alert_mean'] = df_final['hdx_alert_level'].apply(get_mean_hdx)

df_final['hdx_medium_count'] = df_final['hdx_alert_level'].apply(get_medium_count_hdx)
df_final['hdx_high_count'] = df_final['hdx_alert_level'].apply(get_high_count_hdx)

df_final['hdx_med_high_count'] = df_final['hdx_alert_level'].apply(count_hdx_med_high)
df_final['hdx_3m_sum'] = df_final.groupby(level='iso3')['hdx_med_high_count'].transform(lambda x: x.rolling(3).sum())

#### Severity Index

In this case we only have the column `inform_severity_index`, so we are going to extract the variables from this one.

In [ ]:
df_final['severity_6m_avg'] = df_final.groupby(level='iso3')['inform_severity_index'].transform(lambda x: x.rolling(6).mean())
df_final['severity_3m_avg'] = df_final.groupby(level='iso3')['inform_severity_index'].transform(lambda x: x.rolling(3).mean())
df_final['severity_lag1'] = df_final.groupby('iso3')['inform_severity_index'].shift(1)
df_final['severity_lag2'] = df_final.groupby('iso3')['inform_severity_index'].shift(2)

NameError: name 'df_final' is not defined

#### CERF SIGNALS

We know from the CERF that depending on the state of a country in a determinate moment, it's susceptible of having one type of conflict or another (hard onset or protracted). The important thing is that depending on the state, the early signals that indicates an incoming crisis are different. So the first thing that we have to do is classify each county x month to one of the two states, and then, depending on it, check which signals are beeing seen. 

In order to classify a country x month in a state, we use the CERF logic, saying:

"A conflict is classified as a “protracted crisis” when EconAI’s risk score remains consistently above 0.6 over a period of 12 months."

Then, the early signals depending on the state are:

1. Protracted: 
- EconAI’s risk score increases by ≥0.1 over 3 consecutive months while already >0.7; OR
- Monthly displacement or fatalities increase by ≥1.5 times rolling 6-month average for 2 consecutive months; OR
- There are ≥3 Medium or High HDX signals occur within 90 days.

2. Hard onset:
- EconAI’s risk score increases by ≥0.3 within 2 months (this will typically also be confirmed by the issuance of an HDX signal the month of the significant increase); OR
- Monthly displacement increases by > 3 times the rolling 6-month average (this will typically be confirmed by a sharp increase in risk score). 

In instances where displacement data is missing, such as Armenia, the fatalities trend can be used instead.


In [ ]:
# 1. CLASSIFY THE STATE (Protracted vs Hard Onset)

# We use rolling sum of 12 on the boolean column. If sum is 12, it was True for 12 consecutive months.
df_final['is_protracted'] = df_final.groupby(level='iso3')['risk_gt_06'].transform(
    lambda x: x.rolling(window=12, min_periods=12).sum() == 12
).astype(int)

df_final['state'] = np.where(df_final['is_protracted']==1, 'Protracted', 'Hard onset')

In [ ]:
# 3. PROTRACTED SIGNALS EVALUATION

# P1: Risk increases by >= 0.1 over 3 months while already > 0.7
df_final['p_sig1'] = (df_final.groupby(level='iso3')['risk_3'].diff(3) >= 0.1) & (df_final['risk_3'] > 0.7)

# P2: Displacement or fatalities >= 1.5x their 6-month average for 2 consecutive months
spike_1_5x = (df_final['monthly_displacement'] >= 1.5 * df_final['disp_6m_avg']) | (df_final['fatalities'] >= 1.5 * df_final['fat_6m_avg'])

# Agrupamos la Serie directamente usando su propio índice (level='iso3')
df_final['p_sig2'] = spike_1_5x.groupby(level='iso3').transform(
    lambda x: x.rolling(2).sum() == 2
)
# P3: >= 3 Medium or High HDX signals within 90 days
df_final['p_sig3'] = df_final['hdx_3m_sum'] >= 3

# Combine Protracted signals (True if any is True)
df_final['protracted_signal'] = df_final['p_sig1'] | df_final['p_sig2'] | df_final['p_sig3']


In [ ]:
# 4. HARD ONSET SIGNALS EVALUATION

# H1: Risk increases by >= 0.3 within 2 months
df_final['h_sig1'] = df_final.groupby(level='iso3')['risk_3'].diff(2) >= 0.3

# H2: Displacement > 3x 6-month avg. If missing, use fatalities > 3x 6-month avg.
disp_spike_3x = df_final['monthly_displacement'] > 3 * df_final['disp_6m_avg']
fat_spike_3x = df_final['fatalities'] > 3 * df_final['fat_6m_avg']

# np.where lets us use the fatality logic ONLY when displacement is NaN
df_final['h_sig2'] = np.where(df_final['monthly_displacement'].isna(), fat_spike_3x, disp_spike_3x)

# Combine Hard Onset signals
df_final['hard_onset_signal'] = df_final['h_sig1'] | df_final['h_sig2']


In [ ]:
# 5. FINAL EARLY SIGNAL COLUMN

# Apply the corresponding signal based on the state of the country that month
df_final['early_signal'] = np.where(
    df_final['state'] == 'Protracted',
    df_final['protracted_signal'],
    df_final['hard_onset_signal']
).astype(int) # Convert True/False to 1/0

# Let's check the distribution!
print(df_final['state'].value_counts())
print(f"\nTotal early signals detected: {df_final['early_signal'].sum()}")

df_final = df_final.drop(columns=['state'])

state
Hard onset    13946
Protracted     1542
Name: count, dtype: int64

Total early signals detected: 685


In [ ]:
df_final.head()

risk_3   risk_12  logfat_risk_3  logfat_risk_12  INFORM  \
iso3 month                                                                   
AFG  2019-01-01  0.997366  1.000000       8.554694        9.716417     7.7   
     2019-02-01  0.993118  0.997806       8.489241        9.786673     7.7   
     2019-03-01  0.994183  0.997635       8.551225        9.668575     7.7   
     2019-04-01  0.992014  0.997427       8.497727        9.726764     7.7   
     2019-05-01  0.993915  0.998244       8.677351        9.830026     7.8   

                  VU   CC   HA  fatalities  event_count  ...  \
iso3 month                                               ...   
AFG  2019-01-01  7.1  7.5  8.7      3255.0         31.0  ...   
     2019-02-01  7.1  7.5  8.7      2501.0         28.0  ...   
     2019-03-01  7.1  7.5  8.7      3159.0         31.0  ...   
     2019-04-01  7.1  7.5  8.7      2803.0         30.0  ...   
     2019-05-01  7.2  7.5  8.8      3404.0         31.0  ...   

                hdx_med_high_count hdx_3m_sum  p_sig1  p_sig2  p_sig3  \
iso3 month                                                              
AFG  2019-01-01                  0        NaN   False   False   False   
     2019-02-01                  0        NaN   False   False   False   
     2019-03-01                  0        0.0   False   False   False   
     2019-04-01                  0        0.0   False   False   False   
     2019-05-01                  0        0.0   False   False   False   

                 protracted_signal  h_sig1  h_sig2  hard_onset_signal  \
iso3 month                                                              
AFG  2019-01-01              False   False   False              False   
     2019-02-01              False   False   False              False   
     2019-03-01              False   False   False              False   
     2019-04-01              False   False   False              False   
     2019-05-01              False   False   False              False   

                 early_signal  
iso3 month                     
AFG  2019-01-01             0  
     2019-02-01             0  
     2019-03-01             0  
     2019-04-01             0  
     2019-05-01             0  

[5 rows x 33 columns]

In [ ]:
display(pd.DataFrame(df_final.columns, columns=['Column Name']))

,Column Name
0,risk_3
1,risk_12
2,logfat_risk_3
3,logfat_risk_12
4,INFORM
5,VU
6,CC
7,HA
8,fatalities
9,event_count


In [ ]:
df_final.columns.tolist()

In [ ]:
df_final.to_parquet("../data_clean/final_data.parquet")
print("Done! Your final_data.parquet is perfectly built and optimized with ALL strategies.")

#### CHECKS ON THE FINAL DATASET

In [151]:
display(pd.DataFrame(df_final.columns, columns=['Column Name']))

,Column Name
0,risk_3
1,risk_12
2,logfat_risk_3
3,logfat_risk_12
4,INFORM
...,...
824,pca_acled_mean_3
825,pca_acled_max_3
826,pca_acled_mean_4
827,pca_acled_max_4


In [152]:
print(df_final.columns.tolist())

['risk_3', 'risk_12', 'logfat_risk_3', 'logfat_risk_12', 'INFORM', 'VU', 'CC', 'HA', 'fatalities', 'event_count', 'notes_acled', 'hdx_alert_level', 'hdx_value', 'monthly_displacement', 'inform_severity_index', 'rolling_3m_displacements', 'allocation-eligible', 'target_2m', 'start_conflict', 'risk_gt_06', 'is_protracted', 'disp_6m_avg', 'fat_6m_avg', 'hdx_med_high_count', 'hdx_3m_sum', 'p_sig1', 'p_sig2', 'p_sig3', 'protracted_signal', 'h_sig1', 'h_sig2', 'hard_onset_signal', 'early_signal', 'monthly_displacement_lag1', 'monthly_displacement_lag2', 'fatalities_lag1', 'risk_3_lag1', 'risk_12_lag1', 'logfat_risk_3_lag1', 'logfat_risk_12_lag1', 'hdx_alert_max', 'hdx_alert_sum', 'hdx_alert_mean', 'hdx_medium_count', 'hdx_high_count', 'acled_disp_score_max', 'acled_disp_score_mean', 'acled_disp_events_count', 'acled_total_events', 'acled_disp_events_ratio', 'emb_acled_mean_0', 'emb_acled_max_0', 'emb_acled_mean_1', 'emb_acled_max_1', 'emb_acled_mean_2', 'emb_acled_max_2', 'emb_acled_mean_3',

In [155]:
df_check = df_final.reset_index() if isinstance(df_final.index, pd.MultiIndex) else df_final.copy()
df_check['month'] = pd.to_datetime(df_check['month'])

# Count unique ISO3 and Months
num_countries = df_check['iso3'].nunique()
num_months = df_check['month'].nunique()

print("=" * 50)
print(f"Dataset statistics:")
print(f"   -> Number of unique countries (iso3): {num_countries}")
print(f"   -> Total number of unique months:  {num_months}")
print("=" * 50)

# Check if there are Gaps (missing months) in the global timeline
min_date = df_check['month'].min()
max_date = df_check['month'].max()
expected_range = pd.date_range(start=min_date, end=max_date, freq='MS') # 'MS' = Month Start
missing_months = [m.strftime('%Y-%m-%d') for m in expected_range if m not in df_check['month'].values]

print(f"Temporal analysis (From {min_date.strftime('%Y-%m')} to {max_date.strftime('%Y-%m')}):")
print(f"   -> Expected months: {len(expected_range)}")

if len(missing_months) == 0:
    print("   -> PERFECT: No temporal gaps found in the global dataset. All months are covered.")
else:
    print(f"   -> ALERT: {len(missing_months)} months are missing in the global dataset!")
    print(f"   -> Missing months: {missing_months}")
print("=" * 50)

Dataset statistics:
   -> Number of unique countries (iso3): 176
   -> Total number of unique months:  88
Temporal analysis (From 2019-01 to 2026-04):
   -> Expected months: 88
   -> PERFECT: No temporal gaps found in the global dataset. All months are covered.


/var/folders/n9/kv5pq1015h5cxdk8s76mbdxw0000gn/T/ipykernel_1122/62802408.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_check = df_final.reset_index() if isinstance(df_final.index, pd.MultiIndex) else df_final.copy()
/var/folders/n9/kv5pq1015h5cxdk8s76mbdxw0000gn/T/ipykernel_1122/62802408.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_check = df_final.reset_index() if isinstance(df_final.index, pd.MultiIndex) else df_final.copy()


### PLOT THE RESULTS:

In [154]:
df = pd.read_parquet("../data_clean/final_data.parquet")
df = df.reset_index()

In [156]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go

def plot_full_crisis_timeline(df, cerf_path, country_code):
    # 1. PREPARAR EL DATAFRAME PRINCIPAL
    if 'iso3' not in df.columns:
        df = df.reset_index()
        
    df_plot = df[df['iso3'] == country_code].copy()
    df_plot['month'] = pd.to_datetime(df_plot['month'])
    df_plot = df_plot.sort_values('month')
    
    if df_plot.empty:
        print(f"No hay datos en el DataFrame principal para el país: {country_code}")
        return

    # Transformación Log(1 + x) para desplazamientos
    df_plot['disp_log1p'] = np.log1p(df_plot['monthly_displacement'].fillna(0))

    # Extraer las fechas de alerta target_2m (marcadas a mitad de mes)
    if 'target_2m' in df_plot.columns:
        warning_months = df_plot[df_plot['target_2m'] == 1]['month']
        warning_dates = warning_months + pd.Timedelta(days=0)
    else:
        warning_dates = pd.Series(dtype='datetime64[ns]')
    
    # Fechas donde target_2m es NaN
    if 'target_2m' in df_plot.columns:
        nan_months = df_plot[df_plot['target_2m'].isna()]['month']
        nan_dates = nan_months + pd.Timedelta(days=0)
    else:
        nan_dates = pd.Series(dtype='datetime64[ns]')

    # 3. CREAR LA FIGURA INTERACTIVA
    fig = go.Figure()

    # --- B. Desplazamientos (Línea simple morada - Puesta en y1 para que NO desaparezca) ---
    fig.add_trace(go.Scatter(
        x=df_plot['month'],
        y=df_plot['disp_log1p'],
        customdata=df_plot['monthly_displacement'], 
        mode='lines', 
        name='Displacements [log(1+x)]',
        line=dict(color='purple', width=2.5),
        yaxis='y1',
        hovertemplate='<b>Date:</b> %{x|%b %Y}<br><b>Displacements (Real):</b> %{customdata:,.0f}<br><b>log(1+x):</b> %{y:.2f}<extra></extra>'
    ))

    # --- D. Zonas de Conflicto (Fondo sombreado salmón) ---
    if 'allocation-eligible' in df_plot.columns:
        conflict_months = df_plot[df_plot['allocation-eligible'] == 1]['month']
        for c_month in conflict_months:
            fig.add_vrect(
                x0=c_month - pd.Timedelta(days=15), 
                x1=c_month + pd.Timedelta(days=15),
                fillcolor="salmon",
                opacity=0.2,
                layer="below",
                line_width=0,
            )
        # Leyenda del conflicto
        fig.add_trace(go.Scatter(
            x=[None], y=[None], mode='lines', name='Allocation-eligible Zone',
            line=dict(color='salmon', width=10), opacity=0.3, yaxis='y1'
        ))

    # --- NUEVO: Condición >50k Desplazamientos (Fondo azul clarito) ---
    # Comprueba que tu columna se llame 'target'. Si se llama 'target_4m', cámbialo aquí.
    target_col = 'target' if 'target' in df_plot.columns else None
    if target_col:
        target_months = df_plot[df_plot[target_col] == 1]['month']
        for t_month in target_months:
            fig.add_vrect(
                x0=t_month - pd.Timedelta(days=15), 
                x1=t_month + pd.Timedelta(days=15),
                fillcolor="lightblue",
                opacity=0.3,
                layer="below",
                line_width=0,
            )
        fig.add_trace(go.Scatter(
            x=[None], y=[None], mode='lines', name='>50k Displacements (Target)',
            line=dict(color='lightblue', width=10), opacity=0.4, yaxis='y1'
        ))

    # --- F. Líneas conflict_2m (Líneas verticales rojas a rayas) ---
    if not warning_dates.empty:
        for w_date in warning_dates:
            fig.add_shape(
                type="line",
                x0=w_date, x1=w_date,
                y0=0, y1=1,
                xref="x", yref="paper",
                line=dict(width=2.5, dash="dash", color="red"),
            )
        # Leyenda de Alertas
        fig.add_trace(go.Scatter(
            x=[None], y=[None], mode='lines', name='Warning (1-2m Pre-Conflict)',
            line=dict(color='red', width=2.5, dash='dash'), yaxis='y1'
        ))

    # --- G. Líneas target_2m = NaN (gris oscuro a rayas) ---
    if not nan_dates.empty:
        for n_date in nan_dates:
            fig.add_shape(
                type="line",
                x0=n_date, x1=n_date,
                y0=0, y1=1,
                xref="x", yref="paper",
                line=dict(
                    width=2,
                    dash="dash",
                    color="dimgray"
                ),
            )

        # Leyenda
        fig.add_trace(go.Scatter(
            x=[None], y=[None],
            mode='lines',
            name='Allocation-eligible (target = NaN)',
            line=dict(
                color='dimgray',
                width=2,
                dash='dash'
            ),
            yaxis='y1'
        ))

    # --- 4. CONFIGURACIÓN DEL LAYOUT (Eliminados los ejes invisibles) ---
    fig.update_layout(
        title=dict(
            text=f'<b>Full Crisis Timeline Overview: {country_code}</b>',
            font=dict(size=22),
            x=0.05
        ),
        margin=dict(l=60, r=100, t=110, b=60), 
        height=600,
        plot_bgcolor='white',
        hovermode="closest",
        
        xaxis=dict(
            title=dict(text="<b>Date</b>"),
            showgrid=False,
            dtick="M3",
            tickformat="%b %Y",
            tickangle=45
        ),
        
        # Eje Y1: Ahora son los Desplazamientos
        yaxis=dict(
            title=dict(text="<b>log(1 + Displacements)</b>", font=dict(color="purple")),
            tickfont=dict(color="purple"),
            showgrid=True,
            gridcolor='lightgrey',
            rangemode="tozero"
        ),
        
        legend=dict(
            orientation="h",
            yanchor="bottom", y=1.02,
            xanchor="center", x=0.5,
            bgcolor='rgba(255,255,255,0.8)',
            bordercolor='lightgrey',
            borderwidth=1
        )
    )

    fig.show()

# --- EJECUCIÓN ---
plot_full_crisis_timeline(df_final, cerf_path="../data_clean/cerf_clean.csv", country_code="AFG")

/var/folders/n9/kv5pq1015h5cxdk8s76mbdxw0000gn/T/ipykernel_1122/1853142240.py:8: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df = df.reset_index()
/var/folders/n9/kv5pq1015h5cxdk8s76mbdxw0000gn/T/ipykernel_1122/1853142240.py:8: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df = df.reset_index()


In [1]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go

def plot_full_crisis_timeline(df, cerf_path, country_code):
    # 1. PREPARAR EL DATAFRAME PRINCIPAL
    if 'iso3' not in df.columns:
        df = df.reset_index()
        
    df_plot = df[df['iso3'] == country_code].copy()
    df_plot['month'] = pd.to_datetime(df_plot['month'])
    df_plot = df_plot.sort_values('month')
    
    if df_plot.empty:
        print(f"No hay datos en el DataFrame principal para el país: {country_code}")
        return

    # Transformación Log(1 + x) para desplazamientos
    df_plot['disp_log1p'] = np.log1p(df_plot['monthly_displacement'].fillna(0))

    # 2. PREPARAR EL DATAFRAME DE CERF
    try:
        cerf = pd.read_csv(cerf_path)
        cerf = cerf[cerf['iso3'] == country_code].copy()
        cerf['Allocation Date'] = pd.to_datetime(cerf['Allocation Date'], format='ISO8601', errors='coerce')
        cerf = cerf.dropna(subset=['Allocation Date'])
    except FileNotFoundError:
        print(f"No se encontró el archivo CERF en {cerf_path}.")
        cerf = pd.DataFrame()

    # Extraer las fechas de alerta target_2m (marcadas a mitad de mes)
    if 'target_2m' in df_plot.columns:
        warning_months = df_plot[df_plot['target_2m'] == 1]['month']
        warning_dates = warning_months + pd.Timedelta(days=0)
    else:
        warning_dates = pd.Series(dtype='datetime64[ns]')

    # 3. CREAR LA FIGURA INTERACTIVA
    fig = go.Figure()

    # --- A. Fatalidades (Línea simple naranja - y1) ---
    fig.add_trace(go.Scatter(
        x=df_plot['month'],
        y=df_plot['fatalities'],
        mode='lines', 
        name='Fatalities',
        line=dict(color='orange', width=2.5),
        yaxis='y1',
        hovertemplate='<b>Date:</b> %{x|%b %Y}<br><b>Fatalities:</b> %{y:,.0f}<extra></extra>'
    ))

    # --- B. Desplazamientos (Línea simple morada - y2) ---
    fig.add_trace(go.Scatter(
        x=df_plot['month'],
        y=df_plot['disp_log1p'],
        customdata=df_plot['monthly_displacement'], 
        mode='lines', 
        name='Displacements [log(1+x)]',
        line=dict(color='purple', width=2.5),
        yaxis='y2',
        hovertemplate='<b>Date:</b> %{x|%b %Y}<br><b>Displacements (Real):</b> %{customdata:,.0f}<br><b>log(1+x):</b> %{y:.2f}<extra></extra>'
    ))

    # --- C. Riesgo (Línea Negra Punteada - y3) ---
    if 'risk_3' in df_plot.columns:
        fig.add_trace(go.Scatter(
            x=df_plot['month'],
            y=df_plot['risk_3'],
            mode='lines',
            name='Risk Score',
            line=dict(color='black', width=2, dash='dash'),
            yaxis='y3',
            hovertemplate='<b>Date:</b> %{x|%b %Y}<br><b>Risk:</b> %{y:.2f}<extra></extra>'
        ))

    # --- D. Zonas de Conflicto (Fondo sombreado salmón) ---
    if 'allocation-eligible' in df_plot.columns:
        conflict_months = df_plot[df_plot['allocation-eligible'] == 1]['month']
        for c_month in conflict_months:
            fig.add_vrect(
                x0=c_month - pd.Timedelta(days=15), 
                x1=c_month + pd.Timedelta(days=15),
                fillcolor="salmon",
                opacity=0.2,
                layer="below",
                line_width=0,
            )
        # Leyenda del conflicto
        fig.add_trace(go.Scatter(
            x=[None], y=[None], mode='lines', name='Allocation-eligible Zone',
            line=dict(color='salmon', width=10), opacity=0.3, yaxis='y1'
        ))

    # --- E. Líneas CERF (Líneas verticales verdes punteadas) ---
    if not cerf.empty:
        for _, row in cerf.iterrows():
            fig.add_shape(
                type="line",
                x0=row['Allocation Date'], x1=row['Allocation Date'],
                y0=0, y1=1,
                xref="x", yref="paper",
                line=dict(width=2.5, dash="dot", color="mediumseagreen"),
            )
        # Leyenda de CERF
        fig.add_trace(go.Scatter(
            x=[None], y=[None], mode='lines', name='CERF Allocation',
            line=dict(color='mediumseagreen', width=2.5, dash='dot'), yaxis='y1'
        ))

    # --- F. Líneas conflict_2m (Líneas verticales rojas a rayas) ---
    if not warning_dates.empty:
        for w_date in warning_dates:
            fig.add_shape(
                type="line",
                x0=w_date, x1=w_date,
                y0=0, y1=1,
                xref="x", yref="paper",
                line=dict(width=2.5, dash="dash", color="red"),
            )
        # Leyenda de Alertas
        fig.add_trace(go.Scatter(
            x=[None], y=[None], mode='lines', name='Warning (1-2m Pre-Conflict)',
            line=dict(color='red', width=2.5, dash='dash'), yaxis='y1'
        ))

   # --- G. HDX Alert Levels ---

    # --- G. HDX Alert Levels ---

    if 'hdx_alert_level' in df_plot.columns:

        hdx_points = []

        for _, row in df_plot.iterrows():

            alerts = str(row['hdx_alert_level']).lower()

            if 'high concern' in alerts:
                hdx_points.append({
                    'month': row['month'],
                    'level': 'High concern',
                    'color': 'red',
                    'y': 1.08
                })

            if 'medium concern' in alerts:
                hdx_points.append({
                    'month': row['month'],
                    'level': 'Medium concern',
                    'color': 'gold',
                    'y': 1.03
                })

        hdx_alerts = pd.DataFrame(hdx_points)

        if not hdx_alerts.empty:

            fig.add_trace(go.Scatter(
                x=hdx_alerts['month'],
                y=hdx_alerts['y'],

                mode='markers',

                name='HDX Alerts',

                marker=dict(
                    size=7,
                    color=hdx_alerts['color'],
                    symbol='circle',
                    line=dict(width=0.5, color='black')
                ),

                yaxis='y3',

                customdata=hdx_alerts['level'],

                hovertemplate=
                    '<b>HDX SIGNAL</b><br>' +
                    'Date: %{x|%b %Y}<br>' +
                    'Level: %{customdata}<extra></extra>'
            ))

    # --- 4. CONFIGURACIÓN DEL LAYOUT MULTI-EJE ---
    fig.update_layout(
        title=dict(
            text=f'<b>Full Crisis Timeline Overview: {country_code}</b>',
            font=dict(size=22),
            x=0.05
        ),
        margin=dict(l=60, r=100, t=110, b=60), 
        height=600,
        plot_bgcolor='white',
        hovermode="closest",
        
        xaxis=dict(
            domain=[0, 0.85], # Recortamos un poco para que quepa el 3r eje a la derecha
            title=dict(text="<b>Date</b>"),
            showgrid=False,
            dtick="M3",
            tickformat="%b %Y",
            tickangle=45
        ),
        
        # Eje Y1: Izquierda (Fatalidades)
        yaxis=dict(
            title=dict(text="<b>Fatalities</b>", font=dict(color="orange")),
            tickfont=dict(color="orange"),
            showgrid=True,
            gridcolor='lightgrey',
            rangemode="tozero"
        ),
        
        # Eje Y2: Derecha Interna (Desplazamientos Log)
        yaxis2=dict(
            title=dict(text="<b>log(1 + Displacements)</b>", font=dict(color="purple")),
            tickfont=dict(color="purple"),
            anchor="x",
            overlaying="y",
            side="right",
            showgrid=False,
            rangemode="tozero"
        ),
        
        # Eje Y3: Derecha Externa (Risk Score)
        yaxis3=dict(
            title=dict(text="<b>Risk</b>", font=dict(color="black")),
            tickfont=dict(color="black"),
            anchor="free",
            overlaying="y",
            side="right",
            position=1.0,
            range=[0, 1.15],
            showgrid=False
        ),
        
        legend=dict(
            orientation="h",
            yanchor="bottom", y=1.02,
            xanchor="center", x=0.5,
            bgcolor='rgba(255,255,255,0.8)',
            bordercolor='lightgrey',
            borderwidth=1
        )


    )

    fig.show()

# --- EJECUCIÓN ---
plot_full_crisis_timeline(df, cerf_path="../data_clean/cerf_clean.csv", country_code="AFG")

NameError: name 'df' is not defined